In [ ]:
pip install transformers

In [ ]:
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
from google.colab import files
uploaded= files.upload()

Saving samsum-train.csv to samsum-train.csv


In [ ]:
train_data= pd.read_csv("samsum-train.csv")

In [ ]:
from google.colab import files
uploaded= files.upload()

Saving samsum-validation.csv to samsum-validation.csv


In [ ]:
val_data= pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
# random sampling

train_data= train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data= val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
val_data.head()

,id,dialogue,summary
0,13680857,"Edd: wow, did you hear that they're transferri...",Rose and Edd will be transferred to a new depa...
1,13716124,"Tom: Where is the ""Sala del Capitolo""\r\nKevin...","""Sala del Capitolo"" Tom is looking for is in t..."
2,13864418,Patricia: The rowing practice is cancelled!\nK...,The rowing practice is cancelled. A few member...
3,13729340,"Tom: U OK?\r\nAlex: Yeah, pretty good. U?\r\nT...",Tom and Alex had fun last night. They drank a ...
4,13818813,"Patricia: Hello, here's the fair-trade brand I...",Patricia recommends a fair-trade brand she tal...


In [ ]:
# Data Pre-Processing

In [ ]:
import re

def clean_data(text):
  text= re.sub(r"\r\n"," ", text) # lines
  text= re.sub(r"\s+", " ", text) # spaces
  text= re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
  text.strip().lower()
  return text


In [ ]:
train_data["dialogue"]= train_data["dialogue"].apply(clean_data)
train_data["summary"]= train_data["summary"].apply(clean_data)

val_data["dialogue"]= val_data["dialogue"].apply(clean_data)
val_data["summary"]= val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:   Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

In [ ]:

# tokenization

In [ ]:

tokenizer= T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
# convert row data to tokenized input for fine-tuning

def tokenize(data):
  inputs= tokenizer(data["dialogue"], padding= "max_length", max_length= 512, truncation= True)
  targets= tokenizer(data["summary"], padding= "max_length", max_length= 150, truncation= True)

  inputs["labels"]= targets["input_ids"]   # add token ids to input as labels
  return inputs

In [ ]:
train_dataset= train_data.apply(tokenize, axis=1).tolist()      # convert it to list bacause it is compatable with hugging face transformer jiske andr hm train data ko pass krenge.
val_dataset= val_data.apply(tokenize, axis=1).tolist()

In [ ]:

train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
# working with our model

In [ ]:
# NLP =>generation task
# Load the Model.

model= T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# fine- tuning - Fine-tuning is a technique that adapts a pre-trained model to a new task. It uses the knowledge learned from training on a large dataset and applies it to a smaller, task-specific dataset, improving performance while reducing training time.

In [ ]:
if torch.backends.mps.is_available():
  device= torch.device("mps")

elif torch.cuda.is_available():
  device= torch.device("cuda")

else:
  device= torch.device("cpu")

print("device:", device)
model.to(device)

device: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# Training argument
# fine tuning

training_args= TrainingArguments(
    output_dir= "./results",

    num_train_epochs= 6,
    weight_decay=0.01,

    per_device_eval_batch_size= 8,
    eval_strategy= "epoch",
    save_strategy= "epoch",

    warmup_steps= 500
)

In [ ]:
# The Trainer class provides an API for feature-complete training in PyTorch. designed to simplify the process of training and fine-tuning transformer-based models.
# The Trainer class automates the entire training loop, encompassing:

# Forward Pass: Computes model predictions.
# Backward Pass: Calculates gradients and updates model weights.
# Optimization: Applies optimization algorithms to adjust model parameters.
# This automation reduces the need for custom training scripts, thereby minimizing the potential for errors and streamlining the development process.

trainer= Trainer(          # fine tuning
    model= model,
    args= training_args,
    train_dataset= train_dataset,
    eval_dataset= val_dataset
)

In [ ]:
# train the model
# fine Tuning.

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
# steps to use any model like hugging face:
# 1. load the model
# 2. fine tune the model
# 3. save the model

In [ ]:
# save the model

model.save_pretrained("/content/final_model")
tokenizer.save_pretrained("/content/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/final_model/tokenizer_config.json',
 '/content/final_model/tokenizer.json')

In [ ]:
# when we want to use same model and tokenizer

model= T5ForConditionalGeneration.from_pretrained("/content/final_model")
tokenizer= T5Tokenizer.from_pretrained("/content/final_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
# Test the core logic for summarization

def summarize_dialogue(dialogue):
  # before summarization od dialogue, we have to clean the data
  dialogue= clean_data(dialogue)

  # tokenize
  inputs= tokenizer(
      dialogue,
      padding= "max_length",
      max_length= 512,
      truncation= True,
      return_tensors= "pt"
  )

  # generate the summary which is token ids
  targets= model.generate(
      input_ids= inputs["input_ids"],
      attention_mask= inputs["attention_mask"],
      max_length= 150,
      num_beams= 4,        # means hmara jo tranformer hoga vo 4 different sequences of output genrerate krege and finally hme vo summary dega jo best h.
      early_stopping= True

  )

  # convert token ids to text i.e. do decoding
  summary= tokenizer.decode(targets[0], skip_special_tokens= True)
  return summary


In [ ]:
text_dialogue="""
Meera: You look worried. Is everything okay?
Sonia: I'm having trouble completing my machine learning assignment.
Meera: What part are you struggling with?
Sonia: My model's accuracy is very low, even though the code runs without errors.
Meera: Did you check whether your dataset contains missing values?
Sonia: Yes, I handled the missing values yesterday.
Meera: What about feature scaling?
Sonia: I haven't tried that yet. I'm using a model that may benefit from scaled features.
Meera: You should try StandardScaler and compare the results with your current model.
Sonia: That makes sense. Should I also try another algorithm?
Meera: Yes. Train a second model and compare their accuracy, precision, and recall.
Sonia: I'll do that. If the results improve, I'll update my report.
Meera: Good. Also make sure you explain why you selected the final model.
Sonia: Thanks. I'll work on it tonight.
"""

summary= summarize_dialogue(text_dialogue)

print("summary: ", summary)

summary:  Meera's model's accuracy is very low, even though the code runs without errors.
